# Evaluacion de un evento real contra los umbrales q90/q95/q99

Compara la precipitacion observada durante un evento especifico contra los umbrales climatologicos
calculados en `01_umbrales_q90_q95_q99_MSWEP.ipynb`.

Requiere que el evento este en la misma grilla espacial que los umbrales (mismo dataset MSWEP, o
regrillado previamente por el usuario). Si el evento proviene de otra fuente con distinta resolucion,
hay que interpolar antes de comparar (ver seccion opcional al final).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr


## configuracion

In [ ]:
THRESHOLDS_NC = "thresholds_q90_q95_q99.nc"   # salida del notebook 1

# datos del evento real: puede ser un recorte del mismo archivo MSWEP o un dataset distinto
EVENT_PATH_PATTERN = "/path/to/event/*.nc"
EVENT_VAR_NAME = "precipitation"

EVENT_START = "2023-03-10"   # TODO: fechas del evento a evaluar
EVENT_END   = "2023-03-14"


## carga de umbrales y del evento

In [ ]:
thresholds = xr.open_dataset(THRESHOLDS_NC)
# nombre de la variable dentro del netcdf de umbrales (ajustar si difiere)
thr_var = list(thresholds.data_vars)[0]
thresholds = thresholds[thr_var]

event_ds = xr.open_mfdataset(EVENT_PATH_PATTERN, combine="by_coords")
event = event_ds[EVENT_VAR_NAME].sel(time=slice(EVENT_START, EVENT_END))

print(thresholds)
print(event)


## alineacion de grillas

Si `event` y `thresholds` no comparten exactamente la misma grilla (mismo lat/lon), se interpola el
evento a la grilla de los umbrales usando interpolacion bilineal. Si ya coinciden, este paso es un
no-op (`xr.align` con `join="exact"` fallaria si no coinciden, lo cual sirve como chequeo).

In [ ]:
same_grid = (
    event.lat.size == thresholds.lat.size and
    event.lon.size == thresholds.lon.size and
    bool(np.allclose(event.lat.values, thresholds.lat.values)) and
    bool(np.allclose(event.lon.values, thresholds.lon.values))
)

if not same_grid:
    print("grillas distintas: interpolando evento a la grilla de los umbrales (bilineal)")
    event = event.interp(lat=thresholds.lat, lon=thresholds.lon, method="linear")
else:
    print("grillas coincidentes, sin interpolacion")


## exceedencia por dia y por percentil

In [ ]:
exceed = xr.Dataset({
    f"exceed_q{int(q*100)}": (event > thresholds.sel(q=q)).astype(int)
    for q in thresholds.q.values
})

# fraccion del area de estudio que excede cada umbral, por dia
area_frac = exceed.mean(dim=["lat", "lon"])
area_frac.to_dataframe()[[f"exceed_q{int(q*100)}" for q in thresholds.q.values]]


## mapas del evento vs. umbral q95 (ejemplo)

In [ ]:
n_days = event.sizes["time"]
fig, axes = plt.subplots(1, n_days, figsize=(4 * n_days, 4), constrained_layout=True, squeeze=False)
axes = axes[0]

for ax, t in zip(axes, event.time.values, strict=False):
    day = event.sel(time=t)
    day.plot(ax=ax, cmap="Blues", vmin=0)
    # contorno donde se supera q95
    exceed.sel(time=t)["exceed_q95"].plot.contour(ax=ax, levels=[0.5], colors="red", linewidths=1.5)
    ax.set_title(str(np.datetime_as_string(t, unit="D")))

plt.show()


## resumen del evento

In [ ]:
summary = area_frac.max(dim="time").to_pandas()
print("fraccion maxima del area que supero cada umbral durante el evento:")
print(summary)

peak_day = area_frac["exceed_q95"].idxmax(dim="time").values
print(f"dia de maxima extension de exceedencia q95: {peak_day}")
